# M3 video (V1): synthetic clips, per-frame verdicts and smoothing

Turns 200 validation photos (classes in turn, seeded) into 10-second, 15 FPS clips by M1 section 3.4's recipe:
camera path (pan, zoom, rotation, hand shake), motion blur, low light and noise, JPEG, and leaves hiding 30% to 70%
of the snake. The scene is the raw photo; the live step fixes each frame's exposure. The leaves go over the front
end's predicted mask from the notebook 13 kernel, never a SAM mask (R79); a photo without a usable predicted mask gets
them at seeded spots in a fixed central ellipse.

One trained run's model reads the undistorted photo (the clean control) and every frame on the GPU, and each verdict
comes from `decision.decide` at the thresholds `decision_report` fits on that run's own calibration outputs
(`fitted_decision`, with the same `RECALL_BOUND` as the run's decision file, R104b). Three methods turn frames into what the screen shows: the per-frame verdict, an EMA of the
probabilities (alpha 0.3) and 4-of-6 voting on the verdict labels, which shows "unknown" whenever no label fills 4 of
the last 6 frames (R87). Each is scored per clip, for the verdict and for the species: flips, time to the first stable
label and to its confirmation, whether the last stable label is right, how often a dangerous snake's clip ever shows
"not listed", and the same clip's clean-photo verdict beside it.

A box_crop (F1) run reads every frame, and the clean photo, through notebook 13's D1 detector first, as the pipeline
does (R111): the frame's best box is cropped with the same calls `pipeline` makes, and a frame whose box is under the
decision file's `tau_snake` (or that has none) is a "no snake found" frame, smoothed as `pipeline` smooths one. The
detector is `frontends/D1/<winner>/detector.pt` in `PREDS_KERNEL`'s output, named by notebook 13's `winners.json`;
`tau_snake` comes from fitting the decision file with the front end's box confidences. `video_metrics.json` counts the
no-snake frames per clip and overall.

Two schedules are scored from the same frame outputs. `every_frame` reads all 150 frames. `laptop_cpu` plays the clip
live at the per-frame times of the slowest timing run in `data/m3/latency/<LATENCY_BACKEND>/`, measured on Puneet's
laptop CPU (the O6 device), so the model skips the frames that arrive while it is busy, as it would there. A
box_crop run's frames also go through the detector, so its `laptop_cpu` plays the deployed pipeline's one-thread
time from X1's `results/m3/x1_laptop/package.csv` (the deployment row for its encoder with YOLO26n at 320 px,
R113); `run_info.laptop_latency` names the row and figure used.

Outputs go to `/kaggle/working/video/`: `video_metrics.json` (recipe, thresholds, versions, run info, every clip's
generation parameters and the per-schedule, per-method summary), `video_clips.csv` (one row per clip, schedule and
method, added as each clip finishes) and `video_frames.npz` (per clip and schedule: the processed frames, when each
shows, and every method's verdict and top-1, so metrics can be recomputed without the model). Rerunning the run cell
continues where it stopped; a photo that fails is counted and skipped. No frame or clip is saved.

Code: `../src` and `../data` (the runner kernel unpacks them into `/kaggle/working/repo`). The dataset
`serpentguard-indian-snakes-clean` is found with `kagglehub.dataset_download`. The run and the predicted masks come
from the outputs of the kernels `RUN_KERNEL` and `PREDS_KERNEL`, through `kagglehub.notebook_output_download`: a batch
kernel can read another notebook's output only when that notebook is declared as a kernel source. `RUN_ID` names the
run folder (`best.pt`, `config.yaml`, `subset_hash.txt`, `prototypes.npz` and the calibration, validation and unknown
outputs).


In [1]:
RUN_KERNEL = 'dataset:puneetbajajiitm/serpentguard-m3-run-final'
RUN_ID = 'H1_top1_s1'
PREDS_KERNEL = 'puneetbajajiitm/sg-m3-frontends'
N_CLIPS = 200
SEED = 42
LATENCY_BACKEND = "torch_fp32_cpu"  # data/m3/latency/<backend>/: laptop timing runs, the slowest is played for a
                                    # full-frame run; a box_crop run plays X1's deployed pipeline (R113)
LIMIT = None                        # first N clips, for a smoke run
RECALL_BOUND = 'lower'

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the eval normalisation runs through it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
ULTRALYTICS_VERSION = "8.4.202"    # requirements-m3.txt; the D1 detector a box_crop run reads every frame through
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}", f"ultralytics=={ULTRALYTICS_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import timm
import ultralytics

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
print(f"albumentations {albumentations.__version__}, timm {timm.__version__}, ultralytics {ultralytics.__version__}")

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
albumentations 2.0.8, timm 1.0.31, ultralytics 8.4.202


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt, a Global Constraint: the decision report's metrics use it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import shutil
import time
from dataclasses import asdict
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import numpy as np
import pandas as pd
import torch

from serpentguard.config import load_config
from serpentguard.data import ID_TYPES, build_frame, with_files
from serpentguard.decision import DECISION_FILE
from serpentguard.frontend import D1, WINNERS_FILE
from serpentguard.run_data import BOX_CROP, FRONTEND_PREDS_FILE, input_mode
from serpentguard.train import (CALIBRATION_OUTPUTS, CHECKPOINT_FILE, CONFIG_FILE, HASH_FILE, PROTOTYPES_FILE,
                                UNKNOWN_OUTPUTS, VAL_OUTPUTS)
from serpentguard.video import (CLIPS_FILE, FRAMES_FILE, VIDEO_FILE, fitted_decision, round_robin, run_video,
                                laptop_schedule, torch_predictor)

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATASET_PREFIX = "dataset:"  # "dataset:owner/slug": a private dataset copy of a kernel's output, for another account


def kernel_output(handle):
    """Where a kernel's output is: "owner/slug" reads it (a batch kernel only once it is declared a kernel source),
    "dataset:owner/slug" a private dataset copy of it."""
    if handle.startswith(DATASET_PREFIX):
        return Path(kagglehub.dataset_download(handle.removeprefix(DATASET_PREFIX)))
    return Path(kagglehub.notebook_output_download(handle))


DATA = input_root("serpentguard-indian-snakes-clean")
RUN_OUTPUT = kernel_output(RUN_KERNEL)
runs = sorted(path.parent for path in RUN_OUTPUT.rglob(CHECKPOINT_FILE) if path.parent.name == RUN_ID)
assert len(runs) == 1, f"{RUN_KERNEL}'s output must hold one {RUN_ID} folder with {CHECKPOINT_FILE}; found {runs}"
RUN = runs[0]
# the leaves go over the front end's predicted masks, never SAM's (R79)
PREDS_OUTPUT = kernel_output(PREDS_KERNEL)
preds_files = sorted(PREDS_OUTPUT.rglob(FRONTEND_PREDS_FILE))
assert len(preds_files) == 1, f"{PREDS_KERNEL}'s output must hold one {FRONTEND_PREDS_FILE}; found {preds_files}"
PREDS = preds_files[0].parent
DECISION_INPUTS = (CALIBRATION_OUTPUTS, VAL_OUTPUTS, UNKNOWN_OUTPUTS, PROTOTYPES_FILE)
OUT = Path("/kaggle/working/video")

for path in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", RUN / CONFIG_FILE, RUN / HASH_FILE,
             *(RUN / name for name in DECISION_INPUTS)):
    assert path.exists(), f"missing input {path}"
cfg = load_config(RUN / CONFIG_FILE)
assert cfg.run_id == RUN_ID, f"{RUN} holds run {cfg.run_id}"
BOX_CROP_RUN = input_mode(cfg) == BOX_CROP  # F1: every frame goes through the D1 detector first (R111)
DETECTOR = None
if BOX_CROP_RUN:
    winners_files = sorted(PREDS_OUTPUT.rglob(WINNERS_FILE))
    assert len(winners_files) == 1, (f"{PREDS_KERNEL}'s output must hold notebook 13's frontends/{WINNERS_FILE}; "
                                     f"found {winners_files}")
    winners = json.loads(winners_files[0].read_text())
    DETECTOR = winners_files[0].parent / winners[D1]["weights"]
    assert DETECTOR.is_file(), f"the D1 detector {winners[D1]['weights']} is missing from {PREDS_KERNEL}'s output"
LAPTOP_SECONDS, LAPTOP_LATENCY = laptop_schedule(cfg, LATENCY_BACKEND)  # box_crop: X1's deployed pipeline (R113)
assert torch.cuda.is_available(), "no GPU in this session"
DEVICE = torch.device("cuda")
print(f"code {CODE}; run {RUN_ID} at {RUN}; predictions {PREDS}; detector {DETECTOR}; "
      f"{torch.cuda.get_device_name(DEVICE)}")
print(f"laptop step: {np.mean(LAPTOP_SECONDS) * 1000:.1f} ms per frame on average, from {LAPTOP_LATENCY}")

code /kaggle/working/repo/src; run H1_top1_s1 at /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-run-final/H1_top1_s1; predictions /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontend_preds; detector /kaggle/input/notebooks/puneetbajajiitm/sg-m3-frontends/frontends/D1/yolo26n-ft/detector.pt; Tesla T4
laptop step: 84.7 ms per frame on average, from {'file': 'results/m3/x1_laptop/package.csv', 'figure': 'pipeline_ms_1t', 'precision': 'deployment', 'encoder': 'mobilevit_s', 'snakeid_int8': 'fp16', 'size': 320, 'front_end': 'yolo26n-ft@320', 'front_end_precision': 'int8_weights', 'pipeline_ms_1t': 84.68093851115555, 'pipeline_ms_4t': 49.826229573227465, 'latency_device': 'laptop CPU (Apple M4 Pro)'}


In [5]:
# decision_report writes decision_metrics.json next to the outputs it reads, and the input mount is read-only
WORK_RUN = Path("/kaggle/working") / f"{cfg.run_id}_decision"
WORK_RUN.mkdir(parents=True, exist_ok=True)
for name in DECISION_INPUTS:
    shutil.copy2(RUN / name, WORK_RUN / name)
classes = pd.read_csv(DATA / "classes.csv")
metadata = pd.read_csv(DATA / "metadata.csv", dtype=ID_TYPES)
box_conf = None
if BOX_CROP_RUN:  # the front end's box confidences give the decision file its tau_snake (R44)
    preds = pd.read_csv(PREDS / FRONTEND_PREDS_FILE, usecols=["path", "box_conf"])
    box_conf = dict(zip(preds["path"], preds["box_conf"]))
decision = fitted_decision(WORK_RUN, classes, metadata, recall_bound=RECALL_BOUND, box_conf=box_conf)
TAU_SNAKE = json.loads((WORK_RUN / DECISION_FILE).read_text())["no_snake"].get("tau_snake")
assert TAU_SNAKE is not None or not BOX_CROP_RUN, "a box_crop run's decision file must hold tau_snake"
print(decision.thresholds, f"tau_snake {TAU_SNAKE}")

Thresholds(temperature=1.9086784115427977, temperature_who=0.9690096288687304, theta_d=0.05660591583161018, theta_who=inf, tau_unknown=0.40982445157291975, theta_conf=0.360998147721644, theta_v=0.5, theta_genus=0.5) tau_snake 0.102


In [6]:
frontend_preds = pd.read_csv(PREDS / FRONTEND_PREDS_FILE)
frame = with_files(build_frame(metadata, classes, ["val"], frontend_preds=frontend_preds, mask_source="predicted",
                               preds_root=PREDS), DATA)
photos = round_robin(frame, N_CLIPS, SEED)
photos = photos.head(LIMIT) if LIMIT else photos
dangerous = decision.tables.dangerous[photos["class_id"].to_numpy()]
print(f"{len(photos)} clips from {photos['species'].nunique()} classes, {int(dangerous.sum())} of them dangerous; "
      f"{photos['mask_valid'].mean():.0%} with a predicted mask for the leaves (the rest: the central ellipse)")

200 clips from 176 classes, 51 of them dangerous; 96% with a predicted mask for the leaves (the rest: the central ellipse)


In [7]:
started = time.time()
# Ultralytics runs the detector on the GPU when there is one
predict_frame = torch_predictor(RUN, DEVICE, detector=DETECTOR, tau_snake=TAU_SNAKE)
run_info = {"run_kernel": RUN_KERNEL, "run": cfg.run_id, "config": asdict(cfg), "recall_bound": RECALL_BOUND,
            "subset_hash": (RUN / HASH_FILE).read_text().strip(), "device": str(DEVICE),
            "gpu": torch.cuda.get_device_name(DEVICE), "preds_kernel": PREDS_KERNEL, "mask_source": "predicted",
            "detector": None if DETECTOR is None else str(DETECTOR.relative_to(PREDS_OUTPUT)),
            "tau_snake": TAU_SNAKE,
            "laptop_latency": LAPTOP_LATENCY}
schedules = {"every_frame": 0.0, "laptop_cpu": LAPTOP_SECONDS}
report = run_video(photos, predict_frame, decision, OUT, schedules, seed=SEED, run_info=run_info,
                   checkpoint=RUN / CHECKPOINT_FILE)
print(f"{report['clips']} of {report['photos']} clips in {(time.time() - started) / 60:.1f} min, "
      f"{len(report['errors'])} failed; leaves on {report['occluder_regions']}, {report['leaf_cover_below_range']} "
      f"clips under 30% cover; no snake on {report['no_snake']['frames']} of {report['no_snake']['frames_total']} "
      f"frames; wrote {OUT / VIDEO_FILE}, {OUT / CLIPS_FILE} and {OUT / FRAMES_FILE}")
for error in report["errors"]:
    print(error)

200 of 200 clips in 21.4 min, 0 failed; leaves on {'mask': 191, 'centre': 9}, 0 clips under 30% cover; no snake on 1898 of 30000 frames; wrote /kaggle/working/video/video_metrics.json, /kaggle/working/video/video_clips.csv and /kaggle/working/video/video_frames.npz


In [8]:
rows = []
for name, result in report["schedules"].items():
    for method, scores in result["methods"].items():
        verdict, species = scores["verdict"], scores["species"]
        rows.append({"schedule": name, "method": method, "effective FPS": result["effective_fps"],
                     "skipped": result["skipped_share"], "verdict flips (mean)": verdict["mean_flips"],
                     "verdict <= 1 flip": verdict["o6_flips_share"], "verdict stable <= 2 s": verdict["o6_stable_share"],
                     "verdict confirmed <= 2 s": verdict["o6_confirmed_share"],
                     "verdict clip correct": verdict["clip_correct"], "verdict clean correct": verdict["clean_correct"],
                     "verdict clip correct where clean correct": verdict["clip_correct_where_clean_correct"],
                     "species flips (mean)": species["mean_flips"], "species clip correct": species["clip_correct"],
                     "species clean correct": species["clean_correct"],
                     "dangerous ever 'not listed'": scores["dangerous_ever_not_listed"],
                     "dangerous final 'not listed'": scores["dangerous_final_not_listed"]})
summary = pd.DataFrame(rows).set_index(["schedule", "method"])
summary.to_csv(OUT / "summary.csv")
summary.round(3)

effective FPS  skipped  verdict flips (mean)  \
schedule    method                                                      
every_frame per_frame             15.0    0.000                33.230   
            ema                   15.0    0.000                15.485   
            vote_4_of_6           15.0    0.000                 7.665   
laptop_cpu  per_frame             11.9    0.207                26.140   
            ema                   11.9    0.207                12.110   
            vote_4_of_6           11.9    0.207                 6.095   

                         verdict <= 1 flip  verdict stable <= 2 s  \
schedule    method                                                  
every_frame per_frame                0.170                  0.595   
            ema                      0.330                  0.810   
            vote_4_of_6              0.385                  0.945   
laptop_cpu  per_frame                0.190                  0.625   
            ema                      0.360                  0.825   
            vote_4_of_6              0.420                  0.945   

                         verdict confirmed <= 2 s  verdict clip correct  \
schedule    method                                                        
every_frame per_frame                       0.555                 0.285   
            ema                             0.725                 0.305   
            vote_4_of_6                     0.870                 0.295   
laptop_cpu  per_frame                       0.560                 0.285   
            ema                             0.725                 0.295   
            vote_4_of_6                     0.860                 0.295   

                         verdict clean correct  \
schedule    method                               
every_frame per_frame                    0.585   
            ema                          0.585   
            vote_4_of_6                  0.585   
laptop_cpu  per_frame                    0.585   
            ema                          0.585   
            vote_4_of_6                  0.585   

                         verdict clip correct where clean correct  \
schedule    method                                                  
every_frame per_frame                                       0.453   
            ema                                             0.470   
            vote_4_of_6                                     0.462   
laptop_cpu  per_frame                                       0.453   
            ema                                             0.462   
            vote_4_of_6                                     0.462   

                         species flips (mean)  species clip correct  \
schedule    method                                                    
every_frame per_frame                  83.100                 0.070   
            ema                        33.390                 0.120   
            vote_4_of_6                16.015                 0.100   
laptop_cpu  per_frame                  66.165                 0.075   
            ema                        26.545                 0.130   
            vote_4_of_6                12.990                 0.085   

                         species clean correct  dangerous ever 'not listed'  \
schedule    method                                                            
every_frame per_frame                    0.365                        0.294   
            ema                          0.365                        0.078   
            vote_4_of_6                  0.365                        0.078   
laptop_cpu  per_frame                    0.365                        0.294   
            ema                          0.365                        0.078   
            vote_4_of_6                  0.365                        0.039   

                         dangerous final 'not listed'  
schedule    method                                     
every_frame per_fr